# 34 — Skill Normalization Engine
**Goal:** Normalize skill names to canonical taxonomy (ESCO/O*NET).

Extraction (Ch. 33) finds *mentions*; normalization decides what each mention *means*. "ML", "machine learning", and "Machine Learning" are one skill; "AWS" and "Amazon Web Services" are one service. This chapter builds a fallback chain that maps raw strings onto a canonical taxonomy, escalating from exact lookup to fuzzy match to semantic embeddings.

**Why it matters for resumes / ATS:** a normalized skill vocabulary is what makes matching *countable*. An ATS that treats "ML" and "Machine Learning" as different strings under-counts matches against a job description; normalizing both to one canonical name makes scores fair and analytics clean — and it is the only way to compare candidates across thousands of differently-worded resumes.

## 1. Three-Tier Normalization

One normalizer, three escalating strategies — each tier is cheaper and more literal than the last, so the chain only spends effort when it has to:

| Tier | Strategy | Example |
|---|---|---|
| 1 | Exact match against taxonomy | `"Python"` → `"Python (programming language)"` |
| 2 | Fuzzy match (`rapidfuzz`) | `"Tensorflo"` → `"TensorFlow"` (typo tolerance) |
| 3 | Embedding similarity (`sentence-transformers`) | `"ML"` → `"Machine Learning"` (semantic) |

**What the code does:** the cell documents the chain in a printed docstring. Note the docstring quotes a 0.85 fuzzy threshold and a 0.80 embedding threshold, while the actual code in the next cells uses `80` for `fuzz.ratio` and `0.65` for cosine similarity — treat printed thresholds as aspirational and code defaults as truth.

**Why it matters:** the tier also carries confidence meaning — an exact hit is more trustworthy than a fuzzy one, which is more trustworthy than a semantic one — and that ordering flows straight into the schema's `confidence` field in Ch. 39.

In [ ]:
print('''Skill normalization fallback chain:
Tier 1: Exact match -> ESCO/O*NET taxonomy
         "Python" -> "Python (programming language)"
Tier 2: Fuzzy match -> rapidfuzz (threshold 0.85)
         "PyTorch" -> "PyTorch" (typo tolerance)
Tier 3: Embedding similarity -> sentence-transformers (>0.80)
         "ML" -> "Machine Learning"''')

## 2. Building the Normalizer

`SkillNormalizer` wraps a 10-entry canonical taxonomy (including `"ml"` → `"Machine Learning"` and `"aws"` → `"Amazon Web Services"`) and implements tiers 1 and 2: look up the lowercased, stripped raw skill; if absent, fuzzy-match it against the taxonomy keys with `fuzz.ratio`.

**What the code does:** `normalize()` returns `{"normalized", "tier", "confidence"}` — tier 1 hits get `1.0`, tier 2 hits get `score/100`, and unknown skills fall through to tier 0 with `0.5` confidence while keeping the raw string.

**Verified on the sample list:** `"Python"` → `Python (programming language)` (tier 1, conf 1.00); `"pytorch"` → `PyTorch` (tier 1 — case handled by `.lower()`); `"Tensorflo"` → `TensorFlow` (tier 2, conf 0.95); `"ML"` and `"NLP"` → their full forms (tier 1); and `"CloudWhiz"` → unchanged (tier 0, conf 0.50). That last row is the honest case: unknown skills pass through rather than being forced into a wrong bucket.

**Try it:** feed "Pytorch" with wrong casing — only spelling, not casing, costs you a tier.

In [ ]:
from rapidfuzz import fuzz, process

class SkillNormalizer:
    def __init__(self):
        # Canonical skill taxonomy
        self.taxonomy = {
            "python": "Python (programming language)",
            "tensorflow": "TensorFlow",
            "pytorch": "PyTorch",
            "natural language processing": "Natural Language Processing",
            "nlp": "Natural Language Processing",
            "ml": "Machine Learning",
            "machine learning": "Machine Learning",
            "deep learning": "Deep Learning",
            "aws": "Amazon Web Services",
            "gcp": "Google Cloud Platform",
        }
    
    def normalize(self, raw_skill, threshold=80):
        raw = raw_skill.lower().strip()
        
        # Tier 1: exact
        if raw in self.taxonomy:
            return {"normalized": self.taxonomy[raw], "tier": 1, "confidence": 1.0}
        
        # Tier 2: fuzzy
        best = process.extractOne(raw, list(self.taxonomy.keys()), scorer=fuzz.ratio)
        if best and best[1] >= threshold:
            return {"normalized": self.taxonomy[best[0]], "tier": 2, "confidence": best[1]/100}
        
        # Unknown skill
        return {"normalized": raw_skill, "tier": 0, "confidence": 0.5}

n = SkillNormalizer()
for skill in ["Python", "pytorch", "Tensorflo", "ML", "NLP", "CloudWhiz"]:
    result = n.normalize(skill)
    print(f"  '{skill:12s}' -> {result['normalized']:35s} (tier {result['tier']}, conf {result['confidence']:.2f})")

## 3. Embedding-Based Normalization (Tier 3)

The fuzzy tier only rescues *spelling* variants. "ML" vs "Machine Learning" shares almost no characters — only meaning — so it needs semantic similarity: encode both strings with a sentence-transformer and compare the vectors with cosine similarity.

**What the code does:** `EmbeddingNormalizer` subclasses `SkillNormalizer`, encodes the canonical taxonomy once with `all-MiniLM-L6-v2`, then `normalize_embedding()` encodes the raw skill, takes the argmax cosine score, and returns the best canonical text if it clears `threshold=0.65`. Model loading is wrapped so a missing `sentence-transformers` install sets `has_embeddings = False` and degrades to the tier 1/2 fallback instead of failing.

**Expected behavior:** abbreviations and paraphrases ("ML", "NLP", "Cloud computing") map to their canonical forms via embedding proximity, even with near-zero string overlap — at the cost of a ~90 MB model download on first use and slower per-skill latency than the regex/fuzzy tiers.

In [ ]:
from sentence_transformers import SentenceTransformer, util
import re

class EmbeddingNormalizer(SkillNormalizer):
    def __init__(self):
        super().__init__()
        try:
            self.model = SentenceTransformer("all-MiniLM-L6-v2")
            self.canonical_texts = list(self.taxonomy.values())
            self.canonical_embs = self.model.encode(self.canonical_texts)
            self.has_embeddings = True
        except:
            self.has_embeddings = False
    
    def normalize_embedding(self, raw_skill, threshold=0.65):
        if not self.has_embeddings:
            return self.normalize(raw_skill)
        
        emb = self.model.encode(raw_skill)
        scores = util.cos_sim(emb, self.canonical_embs)[0]
        best_idx = scores.argmax().item()
        best_score = scores[best_idx].item()
        
        if best_score >= threshold:
            return {"normalized": self.canonical_texts[best_idx], "tier": 3, "confidence": best_score}
        return {"normalized": raw_skill, "tier": 0, "confidence": 0.5}

en = EmbeddingNormalizer()
if en.has_embeddings:
    for skill in ["Python", "Tensorflow", "ML", "NLP", "Cloud computing"]:
        r = en.normalize_embedding(skill)
        print(f"  '{skill:18s}' -> {r['normalized']:35s} (tier {r['tier']}, conf {r['confidence']:.2f})")
else:
    print("SentenceTransformer not available. Install with: pip install sentence-transformers")

## Summary: Three-tier normalization catches exact matches, typos, and semantic variants.

**Normalization turns noisy extraction into a clean, countable skill vocabulary.**

The tiered fallback is a classic robustness pattern: cheapest and most precise first, most expensive and most flexible last, with each tier reporting its own confidence so downstream consumers can weight the result. Exact and fuzzy tiers run offline in milliseconds; the embedding tier adds semantic coverage for abbreviations and paraphrases when the model is installed.

The output contract — canonical name + tier + confidence — is exactly the `Skill` shape Ch. 39's schema formalizes, so normalized skills flow straight into the final JSON. Next, Ch. 35 applies the same section-scoped thinking to education.